In [0]:
%run ./0-Init

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
import time
import sys

# Buscar apenas tabelas MANAGED AND EXTERNALS(excluindo views)
tables_df = spark.sql(f"""
    SELECT table_catalog,table_schema, table_name
    FROM system.information_schema.tables
    WHERE table_catalog='{var_environment}'
    AND table_type IN ('EXTERNAL','MANAGED')
""")

tables_list = [(row['table_catalog'],row['table_schema'], row['table_name']) for row in tables_df.collect()]
total_tables = len(tables_list)

print(f"Total de tabelas para processar: {total_tables}")
sys.stdout.flush()

success_results = []
error_results = []

# Função para processar cada tabela
def process_table(table_catalog,schema, table):
    full_table_name = f"{table_catalog}.{schema}.{table}"
    start_time = datetime.now()
    
    try:
        # 1. OPTIMIZE (com liquid clustering se já estiver habilitado)
        spark.sql(f"OPTIMIZE {full_table_name}")
        
        # 2. VACUUM (remover arquivos antigos - 7 dias de retenção)
        spark.sql(f"VACUUM {full_table_name} RETAIN 168 HOURS")
        
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        
        return {
            'status': 'SUCCESS',
            'catalog': table_catalog,
            'schema': schema,
            'table': table,
            'full_table_name': full_table_name,
            'start_time': start_time,
            'end_time': end_time,
            'duration_seconds': duration,
            'error_message': None
        }
    except Exception as e:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        error_msg = str(e)
        
        # Handle session errors gracefully
        if "PERMISSION_DENIED" in error_msg or "Local RPC without associated session" in error_msg:
            error_msg = "Session error: Please ensure your cluster/session is active and you have the required permissions."
        
        return {
            'status': 'ERROR',
            'catalog': table_catalog,
            'schema': schema,
            'table': table,
            'full_table_name': full_table_name,
            'start_time': start_time,
            'end_time': end_time,
            'duration_seconds': duration,
            'error_message': error_msg
        }

# Processar tabelas em paralelo
processed = 0
max_workers = int(spark.conf.get("spark.databricks.clusterUsage.numActiveWorkers", "8"))

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Submeter todas as tarefas
    future_to_table = {executor.submit(process_table,table_catalog, schema, table): (table_catalog,schema, table) 
                       for table_catalog, schema, table in tables_list}
    
    # Processar resultados conforme completam
    for future in as_completed(future_to_table):
        table_catalog, schema, table = future_to_table[future]
        result = future.result()
        
        processed += 1
        percentage = (processed / total_tables) * 100
        
        if result['status'] == 'SUCCESS':
            success_results.append(result)
            print(f"✓ [{processed}/{total_tables}] ({percentage:.1f}%) OK: {result['full_table_name']} - {result['duration_seconds']:.2f}s")
        else:
            error_results.append(result)
            print(f"✗ [{processed}/{total_tables}] ({percentage:.1f}%) ERRO: {result['full_table_name']}")
            print(f"  Mensagem: {result['error_message'][:100]}...")
        sys.stdout.flush()

print(f"\nProcessamento concluído!")
print(f"Sucesso: {len(success_results)} tabelas")
print(f"Erro: {len(error_results)} tabelas")
sys.stdout.flush()

# Schema para os dataframes
schema = StructType([
    StructField("catalog", StringType(), True),
    StructField("schema", StringType(), True),
    StructField("table", StringType(), True),
    StructField("full_table_name", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("duration_seconds", StringType(), True),
    StructField("error_message", StringType(), True)
])

# Salvar resultados de SUCESSO em tabela de controle (APPEND)
if success_results:
    success_data = [(r['catalog'], r['schema'], r['table'], r['full_table_name'], 
                     r['start_time'], r['end_time'], str(r['duration_seconds']), r['error_message']) 
                    for r in success_results]
    success_df = spark.createDataFrame(success_data, schema)
    success_df.write.mode("append").saveAsTable(f"{var_environment}.bronze.optimize_vacuum_success_log")
    print(f"\n✓ {len(success_results)} registros salvos em: {var_environment}.bronze.optimize_vacuum_success_log")
    sys.stdout.flush()

# Salvar resultados de ERRO em tabela de controle (APPEND)
if error_results:
    error_data = [(r['catalog'], r['schema'], r['table'], r['full_table_name'], 
                   r['start_time'], r['end_time'], str(r['duration_seconds']), r['error_message']) 
                  for r in error_results]
    error_df = spark.createDataFrame(error_data, schema)
    error_df.write.mode("append").saveAsTable(f"{var_environment}.bronze.optimize_vacuum_error_log")
    print(f"✗ {len(error_results)} registros salvos em: {var_environment}.bronze.optimize_vacuum_error_log")
    sys.stdout.flush()